In [12]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('../scores/S1/perceived_movie/gpt_layer_10/laluna.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['story_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('laluna', 'WER'): np.float64(2.2919088563404566), ('laluna', 'BLEU'): np.float64(4.164183565894594), ('laluna', 'METEOR'): np.float64(3.62231013232706), ('laluna', 'BERT'): np.float32(6.0924754)}


In [13]:
window_zscores = {'subject': [], 'gpt_layer': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in ['S1','S2','S3']:
    for gpt_layer in [6,7,8,9,10]:
        for task in ['laluna']:
            scores = np.load(f'../scores/{subject}/perceived_movie/gpt_layer_{gpt_layer}/{task}.npz', allow_pickle=True)['window_zscores'].item()
            # print(scores['window_zscores'].item())
            window_zscores['subject'].append(subject)
            window_zscores['gpt_layer'].append(gpt_layer)
            window_zscores['WER'].append(scores[(task, 'WER')])
            window_zscores['BLEU'].append(scores[(task, 'BLEU')])
            window_zscores['METEOR'].append(scores[(task, 'METEOR')])
            window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'subject': ['S1',
  'S1',
  'S1',
  'S1',
  'S1',
  'S2',
  'S2',
  'S2',
  'S2',
  'S2',
  'S3',
  'S3',
  'S3',
  'S3',
  'S3'],
 'gpt_layer': [6, 7, 8, 9, 10, 6, 7, 8, 9, 10, 6, 7, 8, 9, 10],
 'WER': [array([-1.93657401e-01, -3.54860432e-01,  5.08162014e-01, -1.08874467e+00,
         -1.68656181e+00, -1.02617250e+00, -2.83337706e-01, -5.59016994e-01,
         -5.65271031e-01,  5.22066004e-02, -1.08000789e+00, -1.27573963e+00,
         -1.57804399e+00, -1.47402102e+00,  3.87298335e-01, -8.32244145e-01,
         -1.54034493e+00,  1.91548220e-01,  4.93625884e-01, -1.04302106e+00,
         -1.74356314e+00, -1.97141975e+00, -6.40866752e-01, -1.31954569e+00,
         -5.40476936e-01, -8.55130347e-01, -2.47100217e-01, -7.42902427e-01,
         -7.91063252e-01, -1.28295818e+00, -1.39457562e+00, -2.11885845e+00,
         -2.23776507e+00, -2.23881897e+00, -1.68105118e+00, -1.80066583e+00,
         -7.75788665e-01, -2.15336282e+00, -1.99884759e+00, -1.93689164e+00,
         -1.71179510e+00, -

In [14]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

331
331
331
=
1689


,subject,gpt_layer,WER,BLEU,METEOR,BERT
0,S1,6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,S1,7,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ..."
2,S1,8,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,S1,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,S1,10,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,S2,6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,S2,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, ..."
7,S2,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,S2,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
9,S2,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [15]:
results_df['BERT'] = results_df['BERT'].apply(np.mean)
to_file = results_df.drop(columns=['WER','BLEU','METEOR']).rename(columns={'BERT':'significantly_decoded'})
to_file

,subject,gpt_layer,significantly_decoded
0,S1,6,0.096677
1,S1,7,0.021148
2,S1,8,0.078550
3,S1,9,0.247734
4,S1,10,0.223565
5,S2,6,0.244713
6,S2,7,0.302115
7,S2,8,0.404834
8,S2,9,0.432024
9,S2,10,0.241692


In [16]:
# gpt_layer_6=np.array(results_df.loc[0, 'BERT']).mean()
# gpt_layer_7=np.array(results_df.loc[1, 'BERT']).mean()
# gpt_layer_8=np.array(results_df.loc[2, 'BERT']).mean()
# gpt_layer_9=np.array(results_df.loc[3, 'BERT']).mean()
# gpt_layer_10=np.array(results_df.loc[4, 'BERT']).mean()
# to_file = pd.DataFrame({'gpt_layer':[6,7,8,9,10], 'significantly_decoded': [gpt_layer_6,gpt_layer_7,gpt_layer_8,gpt_layer_9,gpt_layer_10]})
to_file.to_csv('perceived_movie_gpt_layer.csv', index=False)

to_file

,subject,gpt_layer,significantly_decoded
0,S1,6,0.096677
1,S1,7,0.021148
2,S1,8,0.078550
3,S1,9,0.247734
4,S1,10,0.223565
5,S2,6,0.244713
6,S2,7,0.302115
7,S2,8,0.404834
8,S2,9,0.432024
9,S2,10,0.241692
